# Note 03: Vector Addition

Vector addition is the first complete CUDA program here: it allocates GPU memory, moves data from the CPU, launches a kernel, and copies the result back.

Vector addition is one of the simplest useful CUDA programs.

Suppose we have:

```text
A = [1, 2, 3, 4, 5, 6, 7, 8]

B = [10, 20, 30, 40, 50, 60, 70, 80]
```

We want:

```text
C = A + B

C = [11, 22, 33, 44, 55, 66, 77, 88]
```

On a CPU, we might normally write:

```cpp
for (int i = 0; i < N; i++) {
    c[i] = a[i] + b[i];
}
```

The key observation is that each addition is independent.

For example:

```text
C[0] = A[0] + B[0]
C[1] = A[1] + B[1]
C[2] = A[2] + B[2]
```

None of these calculations depend on each other.

So instead of one CPU thread doing them one after another, we can give different elements to different GPU threads.

```text
Thread 0 → C[0] = A[0] + B[0]

Thread 1 → C[1] = A[1] + B[1]

Thread 2 → C[2] = A[2] + B[2]

...

Thread 7 → C[7] = A[7] + B[7]
```

---

# Complete CUDA Code

```cpp
#include <iostream>
#include <cuda_runtime.h>

__global__ void vec_add(
    const int* a,
    const int* b,
    int* c,
    int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        c[i] = a[i] + b[i];
    }
}

int main() {

    const int N = 8;
    const int bytes = N * sizeof(int);

    // Host arrays
    int h_a[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    int h_b[N] = {10, 20, 30, 40, 50, 60, 70, 80};
    int h_c[N];

    // Device pointers
    int* d_a;
    int* d_b;
    int* d_c;

    // Allocate GPU memory
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy input data from CPU to GPU
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Kernel configuration
    const int threads_per_block = 256;

    const int blocks =
        (N + threads_per_block - 1)
        / threads_per_block;

    // Launch kernel
    vec_add<<<blocks, threads_per_block>>>(
        d_a,
        d_b,
        d_c,
        N
    );

    cudaDeviceSynchronize();

    // Copy result GPU -> CPU
    cudaMemcpy(
        h_c,
        d_c,
        bytes,
        cudaMemcpyDeviceToHost
    );

    // Print result
    for (int i = 0; i < N; i++) {
        std::cout << h_c[i] << " ";
    }

    std::cout << "\n";

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}
```

Expected output:

```text
11 22 33 44 55 66 77 88
```

---

# 1. Host and Device

CUDA programs normally involve two processors:

```text
CPU = Host

GPU = Device
```

The CPU runs the main program.

The GPU runs CUDA kernels.

A simple mental model is:

```text
CPU
 |
 | launches work
 v
GPU
```

The CPU and GPU also have their own memory.

So we usually need to explicitly move data between them.

---

# 2. Host Arrays

These arrays are created inside `main()`:

```cpp
int h_a[N];
int h_b[N];
int h_c[N];
```

The prefix:

```text
h_
```

means:

```text
host
```

So:

```cpp
h_a
```

means:

> Array A stored in CPU memory.

Conceptually:

```text
CPU MEMORY

h_a

[1][2][3][4][5][6][7][8]
```

Likewise:

```text
h_b
```

contains:

```text
[10][20][30][40][50][60][70][80]
```

---

# 3. Device Pointers

We then write:

```cpp
int* d_a;
int* d_b;
int* d_c;
```

The prefix:

```text
d_
```

means:

```text
device
```

But an important detail is:

```cpp
int* d_a;
```

does NOT create an array on the GPU.

It only creates a pointer variable.

A pointer stores an address.

Conceptually:

```text
d_a

┌──────────────────┐
│ address goes here │
└──────────────────┘
```

At this moment, `d_a` does not yet point to valid GPU memory.

---

# 4. Understanding `*`

In:

```cpp
int* d_a;
```

the `*` means:

> `d_a` is a pointer to integers.

It is a variable designed to hold an address.

For example:

```text
d_a = some address
```

The pointer itself is not the array.

It tells us where the array is located.

---

# 5. Allocating GPU Memory

Now we call:

```cpp
cudaMalloc(&d_a, bytes);
```

This asks CUDA:

> Allocate `bytes` amount of memory on the GPU.

Suppose CUDA allocates memory at a device address such as:

```text
0x9000
```

Conceptually:

```text
GPU MEMORY

address 0x9000

[?][?][?][?][?][?][?][?]
```

CUDA then stores that GPU address inside `d_a`.

So after:

```cpp
cudaMalloc(&d_a, bytes);
```

we can imagine:

```text
CPU MEMORY                     GPU MEMORY

d_a                            address 0x9000

┌─────────────┐                [?][?][?][?][?][?][?][?]
│   0x9000    │ ────────────►
└─────────────┘
```

So:

```text
d_a
```

contains the address of GPU memory.

It does not contain the array itself.

---

# 6. Why `&d_a`?

There is an important difference between:

```cpp
d_a
```

and:

```cpp
&d_a
```

`d_a` means:

> Give me the address stored inside the pointer.

`&d_a` means:

> Give me the address where the pointer variable itself is stored.

Suppose:

```text
d_a itself lives at CPU address 0x2000
```

and after `cudaMalloc`:

```text
d_a contains 0x9000
```

Then:

```text
&d_a = 0x2000

d_a  = 0x9000
```

Visual:

```text
CPU MEMORY

address 0x2000

┌──────────────────┐
│ d_a = 0x9000     │
└──────────────────┘
   ↑          │
   │          │
 &d_a        d_a
              │
              │
              ▼

GPU MEMORY

address 0x9000

[?][?][?][?][?][?][?][?]
```

`cudaMalloc` needs:

```cpp
&d_a
```

because CUDA needs to modify the pointer variable.

It needs somewhere to write the newly allocated GPU address.

Conceptually:

```text
cudaMalloc:

1. Find free GPU memory

2. Allocate it

3. Get its address

4. Put that address inside d_a
```

---

# 7. Calculating `bytes`

We write:

```cpp
const int bytes = N * sizeof(int);
```

If:

```text
N = 8
```

and one `int` takes:

```text
4 bytes
```

then:

```text
bytes = 8 × 4

bytes = 32
```

So CUDA allocates enough space for eight integers.

---

# 8. Copying Data from CPU to GPU

After `cudaMalloc`, the GPU allocation exists.

But it still does not contain our actual vector.

CPU memory:

```text
h_a

[1][2][3][4][5][6][7][8]
```

GPU memory:

```text
d_a points here

[?][?][?][?][?][?][?][?]
```

We copy the data using:

```cpp
cudaMemcpy(
    d_a,
    h_a,
    bytes,
    cudaMemcpyHostToDevice
);
```

The general form is:

```cpp
cudaMemcpy(
    destination,
    source,
    number_of_bytes,
    direction
);
```

Here:

```text
destination = d_a

source = h_a
```

and:

```cpp
cudaMemcpyHostToDevice
```

means:

```text
CPU → GPU
```

So CUDA copies the actual array data:

```text
CPU MEMORY                         GPU MEMORY

h_a                               d_a points here

[1][2][3][4][5][6][7][8]
              |
              |
              └───────────────► [1][2][3][4][5][6][7][8]
```

The data is copied.

The pointer itself is not copied.

---

# 9. `h_a` Is an Array, So Why Can `cudaMemcpy` Use It Like an Address?

In C and C++, when an array is passed to a function, its name usually behaves like a pointer to its first element.

So:

```cpp
h_a
```

effectively gives the address of:

```cpp
h_a[0]
```

Conceptually:

```text
h_a

 ↓

[1][2][3][4][5][6][7][8]
 ^
 first element
```

So `cudaMemcpy` knows where the source data starts.

---

# 10. Why Does `cudaMemcpy` Use `d_a` Instead of `&d_a`?

During allocation:

```cpp
cudaMalloc(&d_a, bytes);
```

CUDA needs to change `d_a`.

So it needs:

```text
address of the pointer variable
```

which is:

```cpp
&d_a
```

But during:

```cpp
cudaMemcpy(d_a, h_a, ...)
```

CUDA does not need to modify the pointer.

It needs to use the GPU address already stored inside it.

So:

```cpp
d_a
```

means:

> Copy the data to the GPU memory located at the address stored in `d_a`.

---

# 11. Memory State Before the Kernel

After both copies:

```cpp
cudaMemcpy(d_a, h_a, ...);

cudaMemcpy(d_b, h_b, ...);
```

we have:

```text
CPU MEMORY

h_a
[1][2][3][4][5][6][7][8]

h_b
[10][20][30][40][50][60][70][80]


d_a contains GPU address ────────────────┐
                                         │
d_b contains GPU address ────────────┐   │
                                     │   │
d_c contains GPU address ────────┐   │   │
                                 │   │   │
                                 ▼   ▼   ▼


GPU MEMORY

d_c allocation
[?][?][?][?][?][?][?][?]

d_b allocation
[10][20][30][40][50][60][70][80]

d_a allocation
[1][2][3][4][5][6][7][8]
```

---

# 12. Why Does the Kernel Take Pointer Parameters?

The kernel is:

```cpp
__global__ void vec_add(
    const int* a,
    const int* b,
    int* c,
    int N
)
```

The kernel needs access to arrays stored in GPU memory.

So it needs the addresses of those arrays.

That is why:

```cpp
a
b
c
```

are pointers.

When we launch:

```cpp
vec_add<<<blocks, threads_per_block>>>(
    d_a,
    d_b,
    d_c,
    N
);
```

the GPU addresses stored inside:

```text
d_a
d_b
d_c
```

are passed to:

```text
a
b
c
```

Conceptually:

```text
HOST SIDE

d_a = GPU address A
d_b = GPU address B
d_c = GPU address C


             kernel launch
                   |
                   v


GPU KERNEL

a = same GPU address A
b = same GPU address B
c = same GPU address C
```

So inside the kernel:

```cpp
a[i]
```

means:

> Read element `i` from the GPU memory pointed to by `a`.

And:

```cpp
c[i]
```

means:

> Access element `i` from the GPU memory pointed to by `c`.

---

# 13. Why `const int* a`?

We use:

```cpp
const int* a;
const int* b;
```

because `a` and `b` are inputs.

We only read them.

We do not want the kernel to accidentally modify them.

The output:

```cpp
int* c;
```

is not `const`.

That is because we need to write:

```cpp
c[i] = ...
```

---

# 14. Giving Every Thread Its Own Index

CUDA threads are organized as:

```text
Grid
 |
 +--- Block
 |      |
 |      +--- Thread
 |      +--- Thread
 |
 +--- Block
        |
        +--- Thread
        +--- Thread
```

Inside the kernel:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;
```

calculates a unique global thread index.

For example:

```text
Block 0

Thread 0 → i = 0
Thread 1 → i = 1
Thread 2 → i = 2
Thread 3 → i = 3


Block 1

Thread 0 → i = 4
Thread 1 → i = 5
Thread 2 → i = 6
Thread 3 → i = 7
```

Every thread executes the same kernel code.

But every thread gets a different value of:

```text
i
```

---

# 15. The Actual Vector Addition

The important line is:

```cpp
c[i] = a[i] + b[i];
```

Suppose one thread calculates:

```text
i = 2
```

That thread executes:

```cpp
c[2] = a[2] + b[2];
```

If:

```text
a[2] = 3

b[2] = 30
```

then:

```text
c[2] = 33
```

Another thread may have:

```text
i = 6
```

and performs:

```cpp
c[6] = a[6] + b[6];
```

So conceptually:

```text
Thread 0 → c[0]
Thread 1 → c[1]
Thread 2 → c[2]
Thread 3 → c[3]
...
```

The important CUDA pattern is:

```text
same kernel code
+
different thread index
=
different piece of data
```

---

# 16. Why Do We Need `if (i < N)`?

We write:

```cpp
if (i < N) {
    c[i] = a[i] + b[i];
}
```

because CUDA often launches slightly more threads than we actually need.

Suppose:

```text
N = 1000
```

and:

```text
threads_per_block = 256
```

We need:

```text
4 blocks
```

because:

```text
4 × 256 = 1024 threads
```

But only:

```text
1000 elements
```

exist.

So threads:

```text
1000 to 1023
```

should not access memory.

The check:

```cpp
if (i < N)
```

makes those extra threads do nothing.

---

# 17. Calculating the Number of Blocks

We use:

```cpp
const int threads_per_block = 256;
```

and:

```cpp
const int blocks =
    (N + threads_per_block - 1)
    / threads_per_block;
```

This calculates the number of blocks needed to cover all elements.

It performs ceiling division.

Suppose:

```text
N = 1000
threads_per_block = 256
```

Normal integer division:

```text
1000 / 256 = 3
```

would give only:

```text
3 × 256 = 768 threads
```

which is not enough.

Instead:

```text
(1000 + 256 - 1) / 256

= 1255 / 256

= 4
```

So we launch:

```text
4 × 256 = 1024 threads
```

and the extra threads are ignored using:

```cpp
if (i < N)
```

---

# 18. Why 256 Threads?

`256` is a common practical CUDA block size.

It is not always the best value.

NVIDIA GPUs execute threads in groups called:

```text
warps
```

A warp currently contains:

```text
32 threads
```

So:

```text
256 threads = 8 warps
```

Common block sizes include:

```text
128
256
512
```

because they are multiples of 32.

Later, choosing block size becomes a real performance topic involving:

- warps
- occupancy
- registers
- shared memory
- latency hiding

For now, `256` is simply a common default.

---

# 19. Kernel Launch

The kernel launch is:

```cpp
vec_add<<<blocks, threads_per_block>>>(
    d_a,
    d_b,
    d_c,
    N
);
```

The general syntax is:

```cpp
kernel<<<number_of_blocks, threads_per_block>>>(
    arguments
);
```

For our current example:

```text
N = 8
threads_per_block = 256
```

the number of blocks becomes:

```text
1
```

So the actual launch is effectively:

```cpp
vec_add<<<1, 256>>>(...);
```

Threads:

```text
0 through 7
```

perform useful work.

Threads:

```text
8 through 255
```

fail:

```cpp
if (i < N)
```

and do nothing.

---

# 20. Synchronization

CUDA kernel launches are normally asynchronous.

That means when the CPU launches:

```cpp
vec_add<<<...>>>();
```

the CPU does not necessarily wait for the GPU to finish.

We use:

```cpp
cudaDeviceSynchronize();
```

to tell the CPU:

> Wait here until the GPU has finished its previous work.

Later, unnecessary synchronization can hurt performance.

---

# 21. Copying the Result Back to the CPU

After the kernel finishes:

```text
GPU MEMORY

d_c points here

[11][22][33][44][55][66][77][88]
```

The CPU output array:

```cpp
h_c
```

still does not automatically contain the result.

So we copy it back:

```cpp
cudaMemcpy(
    h_c,
    d_c,
    bytes,
    cudaMemcpyDeviceToHost
);
```

This means:

```text
GPU → CPU
```

Conceptually:

```text
GPU MEMORY                         CPU MEMORY

d_c                               h_c

[11][22][33][44][55][66][77][88]
                  |
                  |
                  └──────────────► [11][22][33][44][55][66][77][88]
```

---

# 22. Freeing GPU Memory

We allocated memory using:

```cpp
cudaMalloc()
```

so we eventually release it using:

```cpp
cudaFree()
```

```cpp
cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_c);
```

We use:

```cpp
d_a
```

instead of:

```cpp
&d_a
```

because `cudaFree` only needs to know:

> Which GPU allocation should be freed?

`d_a` already contains that GPU address.

---

# 23. Full Execution Flow

```text
CPU

h_a = [1 2 3 4 5 6 7 8]

h_b = [10 20 30 40 50 60 70 80]

          |
          |
          v

      cudaMalloc

          |
          v

GPU memory allocated

d_a → [????????????????]
d_b → [????????????????]
d_c → [????????????????]

          |
          |
          v

cudaMemcpy CPU → GPU

          |
          v

GPU

d_a → [1 2 3 4 5 6 7 8]

d_b → [10 20 30 40 50 60 70 80]

d_c → [????????????????]

          |
          |
          v

      Kernel launch

          |
          v

Thread 0 → c[0] = a[0] + b[0]

Thread 1 → c[1] = a[1] + b[1]

Thread 2 → c[2] = a[2] + b[2]

...

Thread 7 → c[7] = a[7] + b[7]

          |
          v

GPU

d_c → [11 22 33 44 55 66 77 88]

          |
          |
          v

cudaMemcpy GPU → CPU

          |
          v

CPU

h_c → [11 22 33 44 55 66 77 88]

          |
          v

        print
```

---

# 24. Most Important Pointer Mental Model

Keep these three things separate:

## Host Array

```cpp
int h_a[N];
```

Actual data stored in CPU memory.

```text
h_a

[1][2][3][4][5][6][7][8]
```

---

## Device Pointer

```cpp
int* d_a;
```

A pointer variable used by the host program.

After `cudaMalloc`, it contains the address of GPU memory.

```text
d_a

┌──────────────┐
│ GPU address  │
└──────────────┘
```

---

## Device Allocation

The actual memory created on the GPU by:

```cpp
cudaMalloc(&d_a, bytes);
```

Conceptually:

```text
d_a
 |
 | contains address
 v

GPU MEMORY

[1][2][3][4][5][6][7][8]
```

The pointer is not the data.

The pointer tells us where the data is.

---

# 25. Core CUDA Pattern

This vector addition already demonstrates the basic CUDA workflow:

```text
Create CPU data
      |
      v
Allocate GPU memory
      |
      v
Copy CPU → GPU
      |
      v
Launch GPU kernel
      |
      v
Each thread works on different data
      |
      v
Copy GPU → CPU
      |
      v
Free GPU memory
```

The operation here is only vector addition.

Later the operation might be:

- matrix multiplication
- reduction
- softmax
- RMSNorm
- attention
- KV-cache operations
- quantization

But the same basic concepts remain:

```text
host
device
pointers
memory allocation
memory copies
threads
blocks
kernels
```

These are the foundations of CUDA programming.

In [ ]:
%%writefile vec_add.cu

#include <iostream>
#include <cuda_runtime.h>

__global__ void vec_add(
    const int* a,
    const int* b,
    int* c,
    int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    const int N = 8;
    const int bytes = N * sizeof(int);

    // Host arrays
    int h_a[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    int h_b[N] = {10, 20, 30, 40, 50, 60, 70, 80};
    int h_c[N];

    // Device pointers
    int* d_a;
    int* d_b;
    int* d_c;

    // Allocate GPU memory
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy inputs CPU -> GPU
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Kernel configuration
    const int threads_per_block = 256;
    const int blocks =
        (N + threads_per_block - 1) / threads_per_block;

    // Launch kernel
    vec_add<<<blocks, threads_per_block>>>(
        d_a,
        d_b,
        d_c,
        N
    );

    cudaDeviceSynchronize();

    // Copy result GPU -> CPU
    cudaMemcpy(
        h_c,
        d_c,
        bytes,
        cudaMemcpyDeviceToHost
    );

    // Print result
    for (int i = 0; i < N; i++) {
        std::cout << h_c[i] << " ";
    }

    std::cout << "\n";

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting vec_add.cu


In [ ]:
!nvcc vec_add.cu -o vec_add
!./vec_add

11 22 33 44 55 66 77 88 
